# ⚡ Notebook 05 — Scalability & Storage Comparison

## What this notebook does
This notebook answers a key project requirement: **is it better to work with raw VCF files or pre-processed Parquet files?**

We measure and compare:
- How long does it take to load each format?
- How much RAM does each use?
- How much disk space does each take?

### Why does this matter?
The full gnomAD dataset (all chromosomes) is **tens of gigabytes**. On a laptop with 8GB RAM, you cannot load it all at once. We need to understand our constraints and plan around them.

### Input
- `data/raw/gnomad.genomes.v4.1.sites.chrY.vcf.bgz` — raw VCF
- `data/raw/variants.parquet` — our processed version

### Output
- A comparison table and charts showing VCF vs Parquet performance

In [ ]:
import time
import os
import psutil
import pandas as pd
import cyvcf2
import matplotlib.pyplot as plt

def get_memory_mb():
    """Returns current RAM usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def file_size_mb(path):
    """Returns file size in MB"""
    return os.path.getsize(path) / 1024 / 1024

print('Setup complete')

In [ ]:
# --- FILE SIZE COMPARISON ---
vcf_path     = '../data/raw/gnomad.genomes.v4.1.sites.chrY.vcf.bgz'
parquet_path = '../data/raw/variants.parquet'

vcf_mb     = file_size_mb(vcf_path)
parquet_mb = file_size_mb(parquet_path)

print(f'Raw VCF file size:     {vcf_mb:.1f} MB')
print(f'Parquet file size:     {parquet_mb:.1f} MB')
print(f'Compression ratio:     {vcf_mb/parquet_mb:.1f}x smaller as Parquet')

In [ ]:
# --- LOAD TIME: Parquet ---
mem_before = get_memory_mb()
start = time.time()

df_parquet = pd.read_parquet(parquet_path)

parquet_time = time.time() - start
parquet_mem  = get_memory_mb() - mem_before

print(f'Parquet load time: {parquet_time:.2f} seconds')
print(f'Parquet RAM usage: {parquet_mem:.0f} MB')

In [ ]:
# --- LOAD TIME: VCF (first 50,000 rows only for fairness) ---
mem_before = get_memory_mb()
start = time.time()

vcf = cyvcf2.VCF(vcf_path)
rows = []
for i, v in enumerate(vcf):
    rows.append({'pos': v.POS, 'af': v.INFO.get('AF')})
    if i >= 49999: break
vcf.close()
df_vcf_sample = pd.DataFrame(rows)

vcf_time = time.time() - start
vcf_mem  = get_memory_mb() - mem_before

print(f'VCF parse time (50k rows): {vcf_time:.2f} seconds')
print(f'VCF RAM usage:             {vcf_mem:.0f} MB')

In [ ]:
# --- SUMMARY TABLE ---
summary = pd.DataFrame({
    'Format'     : ['Raw VCF', 'Parquet'],
    'File Size (MB)' : [vcf_mb, parquet_mb],
    'Load Time (s)'  : [f'{vcf_time:.1f} (50k rows only)', f'{parquet_time:.1f} (full)'],
    'RAM Usage (MB)' : [vcf_mem, parquet_mem],
    'Python Friendly': ['No (needs cyvcf2)', 'Yes (pandas directly)']
})
print('=== Storage Strategy Comparison ===')
print(summary.to_string(index=False))

print('\n✅ Notebook 05 complete. All analysis done!')